# Study 822 — Omega-Ratio Sort — the teardown

The per-leg splits, the Newey-West spread *t*, the Omega-vs-Sharpe-vs-low-vol head-to-head with rank overlaps, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 3873, 'spread_bps': 1.19, 't_nw': 0.76, 't_1s': 0.72, 'hi_bps': 7.93, 'lo_bps': 6.74, 'welch_t': 0.42, 'gross_sharpe': 0.18, 'omega_bps': 1.19, 'omega_t': 0.76, 'sharpe_bps': 1.29, 'sharpe_t': 0.83, 'lowvol_bps': -5.65, 'lowvol_t': -2.92, 'rho_omega_sharpe': 0.996, 'rho_omega_negvol': 0.075, 'placebo_obs': 1.19, 'placebo_mean': 0.087, 'placebo_sd': 0.98, 'placebo_p': 0.12, 'placebo_draws': 1000, 'era_early_bps': 0.29, 'era_early_t': 0.16, 'era_early_n': 1739, 'era_late_bps': 1.92, 'era_late_t': 0.8, 'era_late_n': 2134, 'timer_1_gross': 1.19, 'timer_1_cost': 2.14, 'timer_1_net': -0.95, 'timer_1_t': -0.58, 'timer_5_gross': 1.19, 'timer_5_cost': 10.14, 'timer_5_net': -8.95, 'timer_5_t': -5.44, 'null_mean_t': 0.05, 'null_sd_t': 1.0, 'null_fire': 1, 'planted_t': 9.67, 'planted_welch': 9.49}

## The headline — long-high-Omega / short-low-Omega spread

Daily equal-weight top-30% minus bottom-30% trailing-Omega(0) spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-Omega {R['hi_bps']:+.2f} vs low-Omega {R['lo_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : +1.19 bps/day  NW(10) t = +0.76  one-sample t = +0.72
books         : high-Omega +7.93 vs low-Omega +6.74 bps (Welch t = +0.42)
gross Sharpe  : 0.18 (before cost)


## Does the full gain/loss ratio beat Sharpe? — the head-to-head

Same universe, same dates, same sort machinery. If Omega's extra moments matter, it should out-earn Sharpe and pick different names.

In [3]:
print(f"Omega  (long high / short low): {R['omega_bps']:+.2f} bps  NW t = {R['omega_t']:+.2f}")
print(f"Sharpe (long high / short low): {R['sharpe_bps']:+.2f} bps  NW t = {R['sharpe_t']:+.2f}")
print(f"low-vol(long low / short high): {R['lowvol_bps']:+.2f} bps  NW t = {R['lowvol_t']:+.2f}")
print(f"rank corr  Omega~Sharpe = {R['rho_omega_sharpe']:+.3f}   Omega~(-vol) = {R['rho_omega_negvol']:+.3f}")
print('=> Omega is ~identical to Sharpe (rho +0.996) and does NOT beat it; the low-vol confound is NOT the driver.')

Omega  (long high / short low): +1.19 bps  NW t = +0.76
Sharpe (long high / short low): +1.29 bps  NW t = +0.83
low-vol(long low / short high): -5.65 bps  NW t = -2.92
rank corr  Omega~Sharpe = +0.996   Omega~(-vol) = +0.075
=> Omega is ~identical to Sharpe (rho +0.996) and does NOT beat it; the low-vol confound is NOT the driver.


## Placebo — column-permute the forward returns (1,000 permutations)

In [4]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.3f}")

observed +1.19 bps vs placebo mean +0.087 (sd 0.980) -> right-tail p = 0.120


## Robustness — two eras (split 2018-01-01)

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1739): +0.29 bps  NW t = +0.16
2018-2026 (n=2134): +1.92 bps  NW t = +0.80


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [6]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross +1.19 -> net -0.95 bps/day (cost 2.14/day, t=-0.58)
5 bps one-way: gross +1.19 -> net -8.95 bps/day (cost 10.14/day, t=-5.44)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from omega_ratio import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=822+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=822, n_assets=40, n_days=1500))
print(f"planted (edge=0.0016): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean +0.02 (sd 1.24), |t|>=2 in 0/8


planted (edge=0.0016): NW t = +9.67, Welch t = +9.49


## Verdict

- **Signal — None.** The Keating-Shadwick Omega advantage does **not** materialise on 50 liquid US mega-caps: the long-high-Omega / short-low-Omega spread is **+1.19 bps/day** (NW *t* = **+0.76**) — right sign, insignificant, flat in both eras (*t* = +0.16 / +0.80), placebo p = 0.12. It is **0.996 rank-identical to a plain Sharpe sort** and does not beat it (+1.29 bps) — the extra moments add nothing. The 20-seed synthetic control recovers a *planted* effect cleanly (*t* = +9.67, fires on 1/20 nulls), so this is a true null.
- **Tradability — Mirage.** The book is net-negative at every realistic cost: at 1 bp one-way the friction (2.14 bps/day) already exceeds the 1.19 bps gross edge, net **-0.95 bps/day**; at 5 bps **-8.95 bps/day**.